In [1]:
import os

dirs = [
    "app",
    "app/db",
    "app/models",
    "app/services",
    "app/routes",
    "ingestion",
    "ingestion/loaders",
    "data",
    "data/raw",
    "data/processed",
    "notebooks"
]

for d in dirs:
    os.makedirs(d, exist_ok=True)


In [2]:
files = [
    "app/__init__.py",
    "app/db/__init__.py",
    "app/models/__init__.py",
    "app/services/__init__.py",
    "app/routes/__init__.py",
    "ingestion/__init__.py",
    "ingestion/loaders/__init__.py"
]

for f in files:
    open(f, "w").close()


In [3]:
%%writefile app/db/connection.py
import psycopg2
from psycopg2.extras import RealDictCursor

def get_connection():
    return psycopg2.connect(
        dbname="digimon",
        user="postgres",
        password="CHANGE_ME",
        host="localhost",
        port=5432,
        cursor_factory=RealDictCursor
    )


Writing app/db/connection.py


In [8]:
%%writefile ingestion/cards.py
import requests
from psycopg2.extras import execute_values
from app.db.connection import get_connection

API_URL = "https://digimoncard.io/api-public/search.php"

def fetch_all_cards():
    print("Fetching cards from DigimonCard.io...")

    response = requests.get(API_URL, params={"sort": "card"})
    response.raise_for_status()

    cards = response.json()
    print(f"Fetched {len(cards)} cards.")
    return cards

def ingest_cards(cards):
    conn = get_connection()
    cur = conn.cursor()

    rows = []
    for c in cards:
        rows.append((
            c["cardnumber"],
            c["name"],
            c["color"].split("/") if c.get("color") else [],
            int(c["level"]) if c.get("level") else None,
            c.get("type"),
            c["form"].split("/") if c.get("form") else [],
            c.get("effect"),
            c.get("set_name"),
            True,
            "legal",
            []
        ))

    execute_values(cur, """
        INSERT INTO cards
        (card_id, name, color, level, type, traits, effect_text, set_name, is_legal, ban_status, tags)
        VALUES %s
        ON CONFLICT (card_id) DO NOTHING
    """, rows)

    conn.commit()
    cur.close()
    conn.close()

    print("Card ingestion complete.")


Overwriting ingestion/cards.py


In [5]:

from ingestion.cards import fetch_all_cards, ingest_cards

cards = fetch_all_cards()
    


Fetching cards from DigimonCard.io...
Fetched 8845 cards.


In [8]:
cards[0].keys()

dict_keys(['name', 'type', 'id', 'level', 'play_cost', 'evolution_cost', 'evolution_color', 'evolution_level', 'xros_req', 'color', 'color2', 'digi_type', 'digi_type2', 'digi_type3', 'digi_type4', 'form', 'dp', 'attribute', 'rarity', 'stage', 'artist', 'main_effect', 'source_effect', 'link_requirements', 'link_dp', 'alt_effect', 'series', 'pretty_url', 'date_added', 'tcgplayer_name', 'tcgplayer_id', 'set_name'])

In [9]:
%%writefile schema.sql

-- Drop existing table if resetting schema
DROP TABLE IF EXISTS cards;

-- Main cards table
CREATE TABLE cards (
    card_id TEXT PRIMARY KEY,              -- maps to "id"
    name TEXT,
    type TEXT,
    level INT,
    play_cost INT,
    evolution_cost INT,
    evolution_color TEXT[],
    evolution_level INT,
    xros_req TEXT,
    color TEXT[],
    digi_type TEXT[],
    form TEXT,
    dp INT,
    attribute TEXT,
    rarity TEXT,
    stage TEXT,
    artist TEXT,
    main_effect TEXT,
    source_effect TEXT,
    alt_effect TEXT,
    link_requirements TEXT,
    link_dp INT,
    series TEXT,
    pretty_url TEXT,
    date_added TIMESTAMP,
    tcgplayer_name TEXT,
    tcgplayer_id TEXT,
    set_name TEXT,
    tags TEXT[] DEFAULT '{}'
);

-- Optional: index for faster search by name
CREATE INDEX idx_cards_name ON cards (name);

-- Optional: index for type
CREATE INDEX idx_cards_type ON cards (type);

-- Optional: index for set_name
CREATE INDEX idx_cards_set_name ON cards (set_name);

-- Optional: index for rarity
CREATE INDEX idx_cards_rarity ON cards (rarity);

-- Optional: index for color array
CREATE INDEX idx_cards_color ON cards USING GIN (color);

-- Optional: index for digi_type array
CREATE INDEX idx_cards_digi_type ON cards USING GIN (digi_type);


Writing schema.sql


In [1]:
from ingestion.loaders.ingestion import run_card_ingestion
run_card_ingestion()


Fetching cards from DigimonCard.io...
Fetched 8845 cards.
Card ingestion complete.
Done.


In [3]:
from app.db.connection import get_connection

conn = get_connection()
print("Connected!")
conn.close()


Connected!


In [2]:
from bs4 import BeautifulSoup
import requests

url = "https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt25-dual-revolution-st23-24/"
html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

deck_entries = []

for row in soup.select("table tbody tr"):
    cols = row.find_all("td")
    if not cols:
        continue

    link = cols[0].find("a")
    if not link:
        continue

    deck_entries.append({
        "deck_url": link["href"],
        "deck_name": link.text.strip(),
        "date": cols[1].text.strip(),
        "country": cols[2].text.strip(),
        "author": cols[3].text.strip(),
        "placement": cols[4].text.strip(),
        "tournament": cols[5].text.strip(),
        "host": cols[6].text.strip(),
    })

deck_entries


[]

In [4]:
url = "https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt25-dual-revolution-st23-24/"
html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

deck_entries = []
soup.select("table tbody tr")

[]

In [6]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

html = requests.get(url, headers=headers).text


In [7]:
html

'<!DOCTYPE html>\n<html lang="en-US" prefix="og: https://ogp.me/ns#" class="no-js">\n<head>\n\t<meta charset="UTF-8">\n\t<meta name="viewport" content="width=device-width, initial-scale=1.0">\n\t<link rel="profile" href="https://gmpg.org/xfn/11">\n\t\t<script>\n(function(html){html.className = html.className.replace(/\\bno-js\\b/,\'js\')})(document.documentElement);\n//# sourceURL=twentysixteen_javascript_detection\n</script>\n\n<!-- Search Engine Optimization by Rank Math - https://rankmath.com/ -->\n<title>Decklist JP + CN + EN: BT25 Dual Revolution (+ST23/24) | DIGIMON CARD META</title>\n<meta name="robots" content="follow, index, max-snippet:-1, max-video-preview:-1, max-image-preview:large"/>\n<link rel="canonical" href="https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt25-dual-revolution-st23-24/" />\n<meta property="og:locale" content="en_US" />\n<meta property="og:type" content="article" />\n<meta property="og:title" content="Decklist JP + CN + EN: BT25 Dual Revolution (+ST

In [14]:
pip install playwright

   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/37.9 MB 787.7 kB/s eta 0:00:49
   ---------------------------------------- 0.2/37.9 MB 1.8 MB/s eta 0:00:21
    --------------------------------------- 0.6/37.9 MB 3.2 MB/s eta 0:00:12
    --------------------------------------- 0.9/37.9 MB 3.9 MB/s eta 0:00:10
   - -------------------------------------- 1.3/37.9 MB 4.9 MB/s eta 0:00:08
   -- ------------------------------------- 1.9/37.9 MB 6.2 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/37.9 MB 7.1 MB/s eta 0:00:05
   --- ------------------------------------ 3.1/37.9 MB 7.9 MB/s eta 0:00:05
   --- ------------------------------------ 3.2/37.9 MB 7.4 MB/s eta 0:00:05
   ---- ----------------------------------- 3.8/37.9 MB 7.9 MB/s eta 0:00:05
   ---- ----------------------------------- 4.5/37.9 MB 8.4 MB/s eta 0:00:04
   ----- --

In [16]:
pip install asyncio

Note: you may need to restart the kernel to use updated packages.


In [17]:
import asyncio
from playwright.async_api import async_playwright

async def fetch_page():
    url = "https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt25-dual-revolution-st23-24/"

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url)
        await page.wait_for_selector("table tbody tr")

        html = await page.content()
        return html



In [18]:
html = asyncio.run(fetch_page())
print(len(html))


RuntimeError: asyncio.run() cannot be called from a running event loop

In [15]:
from playwright.sync_api import sync_playwright

url = "https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt25-dual-revolution-st23-24/"

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto(url)
    page.wait_for_selector("table tbody tr")

    html = page.content()


Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

In [12]:
soup.select("table")

[]

In [9]:
deck_entries = []

for row in soup.select("table tbody tr"):
    cols = row.find_all("td")
    if not cols:
        continue

    link = cols[0].find("a")
    if not link:
        continue

    deck_entries.append({
        "deck_url": link["href"],
        "deck_name": link.text.strip(),
        "date": cols[1].text.strip(),
        "country": cols[2].text.strip(),
        "author": cols[3].text.strip(),
        "placement": cols[4].text.strip(),
        "tournament": cols[5].text.strip(),
        "host": cols[6].text.strip(),
    })

In [10]:
deck_entries

[]

In [5]:
html

'403 - Forbidden | Access to this page is forbidden.\n'

In [3]:
link

NameError: name 'link' is not defined

In [19]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import parse_qs, urlparse

url = "https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt25-dual-revolution-st23-24/"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
soup = BeautifulSoup(html, "html.parser")

deck_entries = []

for a in soup.select("a[href^='deckinfo2']"):
    href = a["href"]
    qs = parse_qs(urlparse(href).query)

    deck_entries.append({
        "deck_name": qs.get("dn", [""])[0],
        "date": qs.get("date", [""])[0],
        "country": qs.get("cn", [""])[0],
        "author": qs.get("au", [""])[0],
        "placement": qs.get("pl", [""])[0],
        "tournament": qs.get("tn", [""])[0],
        "host": qs.get("hs", [""])[0],
        "decklist_raw": qs.get("dg", [""])[0],
        "checksum": qs.get("cs", [""])[0],
    })


In [20]:
deck_entries

[{'deck_name': 'TS Marsmon',
  'date': '5/25/2026',
  'country': 'Chile',
  'author': 'Aleph',
  'placement': '1st Place',
  'tournament': 'TB',
  'host': 'Magic4ever(18)',
  'decklist_raw': '4nBT25-001a4nBT25-008a3nBT25-062a4nBT25-064a2nBT24-034a3nBT24-058a4nBT25-014a1nBT24-037a4nBT24-063a1nBT25-020a4nBT25-075a3nBT24-085a2nBT25-091a1nBT25-092a3nBT24-091a3nBT24-100a2nBT25-100a3nBT25-101a3nBT25-102',
  'checksum': '242'},
 {'deck_name': 'BG Imperial',
  'date': '5/25/2026',
  'country': 'Indonesia',
  'author': 'rivalry',
  'placement': '1st Place',
  'tournament': 'TB',
  'host': 'Tcgshop Pontianak(14)',
  'decklist_raw': '4nP-117a4nBT12-021a4nBT12-047a1nBT16-017a1nST9-09a1nEX1-014a4nBT12-022a2nBT12-050a4nBT21-037a4nAD1-011a3nBT16-025a2nST9-06a2nAD1-024a2nBT12-030a1nBT16-027a1nBT20-020a2nBT3-093a4nBT16-085a1nP-104a1nST2-13a1nBT3-103a1nLM-036',
  'checksum': '270'},
 {'deck_name': 'TS Marsmon',
  'date': '5/24/2026',
  'country': 'JP',
  'author': 'Albatros',
  'placement': '1st (7-2)',

In [22]:
%%writefile ingestion/loaders/decks_loader.py

# digimon_meta_scraper.py

import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

# ---------- Data models ----------

@dataclass
class DeckCard:
    card_id: str
    quantity: int

@dataclass
class DeckEntry:
    block_id: str
    deck_name: str
    date: str
    country: str
    author: str
    placement: str
    tournament: str
    host: str
    checksum: str
    cards: List[DeckCard]
    raw_dg: str
    source_url: str


# ---------- Core helpers ----------

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)


def fetch_html(url: str) -> str:
    resp = requests.get(url, headers={"User-Agent": USER_AGENT})
    resp.raise_for_status()
    return resp.text


def parse_deckinfo_link(href: str) -> Dict[str, Any]:
    """
    Parse a single deckinfo2 href into metadata + raw decklist string.
    Example href:
      deckinfo2?dn=TS Marsmon&date=5/25/2026&cn=Chile&au=Aleph&...
    """
    # Ensure we only parse the query part
    parsed = urlparse(href)
    qs = parse_qs(parsed.query if parsed.query else href.split("?", 1)[-1])

    def get(key: str) -> str:
        return qs.get(key, [""])[0]

    return {
        "deck_name": get("dn"),
        "date": get("date"),
        "country": get("cn"),
        "author": get("au"),
        "placement": get("pl"),
        "tournament": get("tn"),
        "host": get("hs"),
        "decklist_raw": get("dg"),
        "checksum": get("cs"),
    }


def decode_decklist(dg: str) -> List[DeckCard]:
    """
    Decode the dg string:
      4nBT25-001a4nBT25-008a3nBT25-062...
    Pattern: {qty}n{card_id}a
    """
    cards: List[DeckCard] = []
    if not dg:
        return cards

    # Find all (quantity, card_id) pairs
    matches = re.findall(r"(\d+)n([A-Z0-9-]+)", dg)
    for qty_str, card_id in matches:
        cards.append(DeckCard(card_id=card_id, quantity=int(qty_str)))
    return cards


# ---------- Scraper for a single meta block page ----------

def scrape_meta_block_page(
    url: str,
    block_id: str,
) -> List[DeckEntry]:
    """
    Scrape a meta block page (e.g. BT25 Dual Revolution) and return all deck entries.
    Uses the deckinfo2 hrefs as the data source.
    """
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    deck_entries: List[DeckEntry] = []

    # All encoded deck links live in <a href="deckinfo2?...">
    for a in soup.select("a[href^='deckinfo2']"):
        href = a.get("href", "").strip()
        if not href:
            continue

        meta = parse_deckinfo_link(href)
        cards = decode_decklist(meta["decklist_raw"])

        deck_entries.append(
            DeckEntry(
                block_id=block_id,
                deck_name=meta["deck_name"],
                date=meta["date"],
                country=meta["country"],
                author=meta["author"],
                placement=meta["placement"],
                tournament=meta["tournament"],
                host=meta["host"],
                checksum=meta["checksum"],
                cards=cards,
                raw_dg=meta["decklist_raw"],
                source_url=url,
            )
        )

    return deck_entries


# ---------- Example orchestrator ----------

META_BLOCKS = [
    {
        "block_id": "ad01",
        "name": "AD-01 Digimon Generation Banlist",
        "url": "https://digimonmeta.com/deck-list/decklist-jp-cn-en-ad-01-digimon-generation-banlist/",
    },
    {
        "block_id": "bt24_ex11",
        "name": "BT24 Time Stranger / EX11 Dawn of Liberator",
        "url": "https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt24-time-stranger-ex11-dawn-of-liberator-decks/",
    },
    {
        "block_id": "ex10_bt23",
        "name": "EX10 Sinister Order / BT23 Hacker's Slumber",
        "url": "https://digimonmeta.com/deck-list/decklist-jp-cn-en-ex10-sinister-order-bt23-hackers-slumber/",
    },
    {
        "block_id": "ex9_bt22",
        "name": "EX9 Versus Royal Knights / BT22 Cyber Eden",
        "url": "https://digimonmeta.com/deck-list/decklist-jp-cn-en-ex9-versus-monsters-bt22-cyber-eden/",
    },
    {
        "block_id": "ex8_bt2_5",
        "name": "EX8 + English Special Booster Ver 2.5",
        "url": "https://digimonmeta.com/deck-list/decklist-english-special-booster-ver-2-5-and-ex8/",
    },
    {
        "block_id": "bt18_bt19",
        "name": "BT18 + BT19 Special Booster",
        "url": "https://digimonmeta.com/deck-list/decklist-english-bt18-and-bt19-special-booster/",
    },
    {
        "block_id": "bt17",
        "name": "BT17 Secret Crisis",
        "url": "https://digimonmeta.com/deck-list/decklist-english-format-bt17-secret-crisis/",
    },
    {
        "block_id": "bt16",
        "name": "BT16 Beginning Observer",
        "url": "https://digimonmeta.com/deck-list/decklist-englishformat-bt16-beginning-observer/",
    },
    {
        "block_id": "bt15",
        "name": "BT15 Exceed Apocalypse",
        "url": "https://digimonmeta.com/deck-list/decklist-english-format-bt15-exceed-apocalypse/",
    },
    {
        "block_id": "bt14_ex5",
        "name": "BT14 + EX5",
        "url": "https://digimonmeta.com/deck-list/decklist-english-format-bt14-ex5-decks/",
    },
    {
        "block_id": "bt13",
        "name": "BT13",
        "url": "https://digimonmeta.com/deck-list/decklist-english-format-bt13-decks/",
    },
    {
        "block_id": "bt12",
        "name": "BT12",
        "url": "https://digimonmeta.com/deck-list/decklist-en-format-bt12-decks/",
    },
    {
        "block_id": "bt11",
        "name": "BT11",
        "url": "https://digimonmeta.com/deck-list/decklist-english-format-bt11-decks/",
    },
    {
        "block_id": "bt10_ex3",
        "name": "BT10 + EX3",
        "url": "https://digimonmeta.com/deck-list/decklist-en-format-bt10-and-ex3-meta-decks/",
    },
    {
        "block_id": "bt9_ex2_st12_st13",
        "name": "BT9 + EX2 + ST12 + ST13",
        "url": "https://digimonmeta.com/deck-list/decklist-english-format-bt9-ex2-st12-st13-meta/",
    },
    {
        "block_id": "bt8_st9_st10",
        "name": "BT8 + ST9 + ST10",
        "url": "https://digimonmeta.com/deck-list/decklist-english-format-bt8-st9-st10-meta/",
    },
    {
        "block_id": "bt6",
        "name": "BT6",
        "url": "https://digimonmeta.com/deck-list/deck-list-english-format-bt6-decks/",
    },
    {
        "block_id": "bt5_bt4_bt3_bt2_bt1",
        "name": "BT1–BT5 English Format",
        "url": "https://digimonmeta.com/deck-list/deck-list-english-format-meta/",
    },
]



def scrape_all_blocks(blocks: Optional[List[Dict[str, str]]] = None) -> List[DeckEntry]:
    if blocks is None:
        blocks = META_BLOCKS

    all_decks: List[DeckEntry] = []
    for block in blocks:
        block_id = block["block_id"]
        url = block["url"]
        print(f"Scraping block {block_id} from {url}...")
        decks = scrape_meta_block_page(url=url, block_id=block_id)
        print(f"  Found {len(decks)} decks.")
        all_decks.extend(decks)
    return all_decks


if __name__ == "__main__":
    decks = scrape_all_blocks()
    print(f"Total decks scraped: {len(decks)}")
    if decks:
        d = decks[0]
        print("Example deck:")
        print(" Name:", d.deck_name)
        print(" Author:", d.author)
        print(" Date:", d.date)
        print(" Country:", d.country)
        print(" Placement:", d.placement)
        print(" Tournament:", d.tournament)
        print(" Host:", d.host)
        print(" Cards:", [(c.card_id, c.quantity) for c in d.cards])


Overwriting ingestion/loaders/decks_loader.py


In [23]:
from ingestion.loaders.decks_loader import scrape_all_blocks

decks = scrape_all_blocks()

Scraping block ad01 from https://digimonmeta.com/deck-list/decklist-jp-cn-en-ad-01-digimon-generation-banlist/...
  Found 313 decks.
Scraping block bt24_ex11 from https://digimonmeta.com/deck-list/decklist-jp-cn-en-bt24-time-stranger-ex11-dawn-of-liberator-decks/...
  Found 508 decks.
Scraping block ex10_bt23 from https://digimonmeta.com/deck-list/decklist-jp-cn-en-ex10-sinister-order-bt23-hackers-slumber/...
  Found 721 decks.
Scraping block ex9_bt22 from https://digimonmeta.com/deck-list/decklist-jp-cn-en-ex9-versus-monsters-bt22-cyber-eden/...
  Found 463 decks.
Scraping block ex8_bt2_5 from https://digimonmeta.com/deck-list/decklist-english-special-booster-ver-2-5-and-ex8/...
  Found 548 decks.
Scraping block bt18_bt19 from https://digimonmeta.com/deck-list/decklist-english-bt18-and-bt19-special-booster/...
  Found 160 decks.
Scraping block bt17 from https://digimonmeta.com/deck-list/decklist-english-format-bt17-secret-crisis/...
  Found 276 decks.
Scraping block bt16 from https://

In [24]:
decks

[DeckEntry(block_id='ad01', deck_name='AlterS', date='5/17/2026', country='Singapore', author='Sam Fong', placement='1st Place', tournament='GAO Prelims(4-0)', host='AGC', checksum='233', cards=[DeckCard(card_id='EX4-003', quantity=4), DeckCard(card_id='EX4-038', quantity=4), DeckCard(card_id='EX4-039', quantity=4), DeckCard(card_id='BT22-017', quantity=2), DeckCard(card_id='AD1-001', quantity=4), DeckCard(card_id='AD1-010', quantity=4), DeckCard(card_id='EX9-012', quantity=4), DeckCard(card_id='EX9-019', quantity=4), DeckCard(card_id='AD1-009', quantity=4), DeckCard(card_id='AD1-012', quantity=4), DeckCard(card_id='EX4-060', quantity=1), DeckCard(card_id='EX9-021', quantity=4), DeckCard(card_id='EX4-061', quantity=2), DeckCard(card_id='ST21-13', quantity=1), DeckCard(card_id='EX9-066', quantity=2), DeckCard(card_id='BT21-102', quantity=2), DeckCard(card_id='BT22-084', quantity=1), DeckCard(card_id='ST20-14', quantity=2), DeckCard(card_id='BT17-095', quantity=1)], raw_dg='4nEX4-003a4nE

In [25]:
len(decks)

5619

In [26]:
import os
print(os.listdir('.'))


['.ipynb_checkpoints', 'app', 'build-project.ipynb', 'data', 'docker-compose.yaml', 'ingestion', 'notebooks', 'schema.sql', 'venv']


In [27]:
from pathlib import Path
for item in Path('.').iterdir():
    print(item)



.ipynb_checkpoints
app
build-project.ipynb
data
docker-compose.yaml
ingestion
notebooks
schema.sql
venv


In [28]:
import os

def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        # Calculate level for indentation
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')

# Use '.' for the current directory
list_files('.')


./
    build-project.ipynb
    docker-compose.yaml
    schema.sql
    .ipynb_checkpoints/
        build_project-checkpoint.ipynb
    app/
        __init__.py
        db/
            connection.py
            __init__.py
            __pycache__/
                connection.cpython-312.pyc
                __init__.cpython-312.pyc
        models/
            __init__.py
        routes/
            __init__.py
        services/
            __init__.py
        __pycache__/
            __init__.cpython-312.pyc
    data/
        processed/
        raw/
    ingestion/
        cards.py
        __init__.py
        loaders/
            decks_loader.py
            ingestion.py
            __init__.py
            __pycache__/
                decks_loader.cpython-312.pyc
                ingestion.cpython-312.pyc
                __init__.cpython-312.pyc
        __pycache__/
            cards.cpython-312.pyc
            __init__.cpython-312.pyc
    notebooks/
    venv/
        pyvenv.cfg
        Includ